# Cafe Sales – Data Cleaning

The raw dataset (`dirty_cafe_sales.csv`) contains 10,000 cafe transactions with missing values,
invalid entries (`ERROR`, `UNKNOWN`) and wrong data types.
**Goal:** produce a clean, analysis-ready dataset.


## 1. Load the data
`ERROR` and `UNKNOWN` are treated as missing values (`NaN`) so they can be handled together with the empty cells.


In [1]:
import pandas as pd
import numpy as np
df = pd.read_csv("../data/dirty/dirty_cafe_sales.csv" , na_values=["ERROR", "UNKNOWN"])

print("DataFrame shape:", df.shape)
print("\n")
print("DataFrame info:")
print(df.info())
print("\n")
print("DataFrame head:")
print(df.head())
print("\n")
print("DataFrame description:")
print(df.describe(include='all'))

print("\n")
print("Unique Price Per Unit values for each Item:")
df.groupby("Item")["Price Per Unit"].unique()


DataFrame shape: (10000, 8)


DataFrame info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Transaction ID    10000 non-null  object 
 1   Item              9031 non-null   object 
 2   Quantity          9521 non-null   float64
 3   Price Per Unit    9467 non-null   float64
 4   Total Spent       9498 non-null   float64
 5   Payment Method    6822 non-null   object 
 6   Location          6039 non-null   object 
 7   Transaction Date  9540 non-null   object 
dtypes: float64(3), object(5)
memory usage: 625.1+ KB
None


DataFrame head:
  Transaction ID    Item  Quantity  Price Per Unit  Total Spent  \
0    TXN_1961373  Coffee       2.0             2.0          4.0   
1    TXN_4977031    Cake       4.0             3.0         12.0   
2    TXN_4271903  Cookie       4.0             1.0          NaN   
3    TXN_7034554   Salad       2

Item
Cake        [3.0, nan]
Coffee      [2.0, nan]
Cookie      [1.0, nan]
Juice       [3.0, nan]
Salad       [5.0, nan]
Sandwich    [4.0, nan]
Smoothie    [4.0, nan]
Tea         [1.5, nan]
Name: Price Per Unit, dtype: object

## 2. Explore the data
### Missing values
Payment Method (~32%) and Location (~40%) have the most missing values.


In [2]:
print("Missing Values:")
print(df.isna().sum())
print("\n")
print("Missing values percentage:")
print((df.isna().sum() / len(df)) * 100)


Missing Values:
Transaction ID         0
Item                 969
Quantity             479
Price Per Unit       533
Total Spent          502
Payment Method      3178
Location            3961
Transaction Date     460
dtype: int64


Missing values percentage:
Transaction ID       0.00
Item                 9.69
Quantity             4.79
Price Per Unit       5.33
Total Spent          5.02
Payment Method      31.78
Location            39.61
Transaction Date     4.60
dtype: float64


### Value counts per column
Checking every column for invalid or unexpected values.


In [3]:
print("Hidden Missing Values:")
for col in df.columns:
    print(f"--- Column: {col} ---")
    print(df[col].value_counts(dropna=False))
    print()

Hidden Missing Values:
--- Column: Transaction ID ---
Transaction ID
TXN_1961373    1
TXN_4831525    1
TXN_1228927    1
TXN_6486912    1
TXN_3447069    1
              ..
TXN_5680238    1
TXN_7766134    1
TXN_2617257    1
TXN_8993132    1
TXN_6170729    1
Name: count, Length: 10000, dtype: int64

--- Column: Item ---
Item
Juice       1171
Coffee      1165
Salad       1148
Cake        1139
Sandwich    1131
Smoothie    1096
Cookie      1092
Tea         1089
NaN          969
Name: count, dtype: int64

--- Column: Quantity ---
Quantity
5.0    2013
2.0    1974
4.0    1863
3.0    1849
1.0    1822
NaN     479
Name: count, dtype: int64

--- Column: Price Per Unit ---
Price Per Unit
3.0    2429
4.0    2331
2.0    1227
5.0    1204
1.0    1143
1.5    1133
NaN     533
Name: count, dtype: int64

--- Column: Total Spent ---
Total Spent
6.0     979
12.0    939
3.0     930
4.0     923
20.0    746
15.0    734
8.0     677
10.0    524
NaN     502
2.0     497
9.0     479
5.0     468
16.0    444
25.0    25

### Duplicates
No duplicate rows or duplicate Transaction IDs were found.


In [4]:
print("Duplicates")
print(df.duplicated().sum())
print("\n")
print("Duplicate Transaction IDs")
print(df["Transaction ID"].duplicated().sum())

Duplicates
0


Duplicate Transaction IDs
0


## 3. Fix data types
Quantity, Price Per Unit and Total Spent are converted to numbers and Transaction Date to datetime.
Invalid values become `NaN`.

In [5]:
print("Wrong Data Types")

pd.to_numeric(df["Quantity"], errors='coerce')
df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")
pd.to_numeric(df["Price Per Unit"], errors='coerce')
df["Price Per Unit"] = pd.to_numeric(df["Price Per Unit"], errors="coerce")
pd.to_numeric(df["Total Spent"], errors='coerce')
df["Total Spent"] = pd.to_numeric(df["Total Spent"], errors="coerce")
pd.to_datetime(df["Transaction Date"], errors='coerce')
df["Transaction Date"] = pd.to_datetime(df["Transaction Date"], errors="coerce")

print(df.dtypes)

Wrong Data Types
Transaction ID              object
Item                        object
Quantity                   float64
Price Per Unit             float64
Total Spent                float64
Payment Method              object
Location                    object
Transaction Date    datetime64[ns]
dtype: object


### Outliers (IQR method)
The only "outliers" are in Total Spent (value 25.0 = 5 Salads × 5.0).
These are valid purchases, so nothing is removed.


In [6]:
numeric_columns=["Quantity", "Price Per Unit", "Total Spent"]
for col in numeric_columns:
    q1=df[col].quantile(0.25)
    q3=df[col].quantile(0.75)
    iqr=q3-q1
    lower_bound=q1-1.5*iqr
    upper_bound=q3+1.5*iqr
    outliers=df[(df[col]<lower_bound) | (df[col]>upper_bound)]
    print(f"--- Column: {col} ---")
    print(f"Number of outliers: {len(outliers)}")
    print(f"Outliers:\n{outliers}\n")
    

--- Column: Quantity ---
Number of outliers: 0
Outliers:
Empty DataFrame
Columns: [Transaction ID, Item, Quantity, Price Per Unit, Total Spent, Payment Method, Location, Transaction Date]
Index: []

--- Column: Price Per Unit ---
Number of outliers: 0
Outliers:
Empty DataFrame
Columns: [Transaction ID, Item, Quantity, Price Per Unit, Total Spent, Payment Method, Location, Transaction Date]
Index: []

--- Column: Total Spent ---
Number of outliers: 259
Outliers:
     Transaction ID   Item  Quantity  Price Per Unit  Total Spent  \
10      TXN_2548360  Salad       5.0             5.0         25.0   
51      TXN_6342161  Salad       5.0             5.0         25.0   
52      TXN_8914892    NaN       5.0             5.0         25.0   
96      TXN_5220895  Salad       5.0             5.0         25.0   
100     TXN_9517146    NaN       5.0             5.0         25.0   
...             ...    ...       ...             ...          ...   
9791    TXN_1232346  Salad       5.0             5.

## 4. Fill numeric values
Every row must satisfy **Total Spent = Quantity × Price Per Unit**.
If one of the three values is missing, it is calculated from the other two.


In [7]:
q,p,t="Quantity" , "Price Per Unit" , "Total Spent"
print(df[[q,p,t]].isna().sum())

df["Total Spent"] = df["Total Spent"].fillna(df["Quantity"] * df["Price Per Unit"])
print(df[[q,p,t]].isna().sum())

df["Price Per Unit"] = df["Price Per Unit"].fillna(df["Total Spent"] / df["Quantity"])
print(df[[q,p,t]].isna().sum())

df["Quantity"] = df["Quantity"].fillna(df["Total Spent"] / df["Price Per Unit"])
print(df[[q,p,t]].isna().sum())





Quantity          479
Price Per Unit    533
Total Spent       502
dtype: int64
Quantity          479
Price Per Unit    533
Total Spent        40
dtype: int64
Quantity          479
Price Per Unit     38
Total Spent        40
dtype: int64
Quantity          38
Price Per Unit    38
Total Spent       40
dtype: int64


### Fill Price from Item
Each item always has the same price in the data (checked with `groupby("Item")["Price Per Unit"].unique()`),
so a missing price can be filled from the item name. Then the formula is applied again.


In [8]:
prices = {"Coffee": 2.0, "Tea": 1.5, "Cookie": 1.0, "Cake": 3.0,
          "Juice": 3.0, "Sandwich": 4.0, "Smoothie": 4.0, "Salad": 5.0}

df[p]=df[p].fillna(df["Item"].map(prices))

df[q]=df[q].fillna(df[t] / df[p])
df[t]=df[t].fillna(df[q] * df[p])

print(df[[q,p,t]].isna().sum())



Quantity          23
Price Per Unit     6
Total Spent       23
dtype: int64


### Drop rows that cannot be recovered
26 rows (0.26%) were missing two or more of Quantity / Price / Total and cannot be recovered, so they are removed.


In [9]:
before = len(df)

df = df.dropna(subset=[q, p, t])
print("Dropped rows:", before - len(df))
print(df[[q, p, t]].isna().sum())

Dropped rows: 26
Quantity          0
Price Per Unit    0
Total Spent       0
dtype: int64


## 5. Fill categorical values
### Item
Prices 1.0, 1.5, 2.0 and 5.0 belong to only one item, so the item can be identified.
Prices 3.0 (Cake/Juice) and 4.0 (Sandwich/Smoothie) are ambiguous → filled with `"Unknown"`.


In [10]:
print("Item missing before:", df["Item"].isna().sum())

unique_price = {1.0: "Cookie", 1.5: "Tea", 2.0: "Coffee", 5.0: "Salad"}
df["Item"] = df["Item"].fillna(df[p].map(unique_price))
df["Item"] = df["Item"].fillna("Unknown")

print("Item missing after:", df["Item"].isna().sum())
print(df["Item"].value_counts())

Item missing before: 963
Item missing after: 0
Item
Coffee      1286
Salad       1270
Cookie      1211
Tea         1206
Juice       1168
Cake        1138
Sandwich    1127
Smoothie    1094
Unknown      474
Name: count, dtype: int64


### Payment Method & Location
Too many values are missing to guess them reliably, so they are filled with `"Unknown"`.


In [11]:
df["Payment Method"] = df["Payment Method"].fillna("Unknown")
df["Location"] = df["Location"].fillna("Unknown")
print(df.isna().sum())


Transaction ID        0
Item                  0
Quantity              0
Price Per Unit        0
Total Spent           0
Payment Method        0
Location              0
Transaction Date    460
dtype: int64


## 6. Transaction Date
460 rows (4.6%) have no date. A date cannot be calculated from other columns, so these rows are removed.


In [12]:
before = len(df)
df = df.dropna(subset=["Transaction Date"])
print("Dropped rows (no date):", before - len(df))
print(df.isna().sum())


Dropped rows (no date): 460
Transaction ID      0
Item                0
Quantity            0
Price Per Unit      0
Total Spent         0
Payment Method      0
Location            0
Transaction Date    0
dtype: int64


## 7. Final checks
- No missing values
- Total Spent = Quantity × Price Per Unit for every row


In [13]:
print("Shape:", df.shape)
print(df.dtypes)
print("Missing total:", df.isna().sum().sum())
print("Q × P == Total:", (df[q] * df[p] == df[t]).all())


Shape: (9514, 8)
Transaction ID              object
Item                        object
Quantity                   float64
Price Per Unit             float64
Total Spent                float64
Payment Method              object
Location                    object
Transaction Date    datetime64[ns]
dtype: object
Missing total: 0
Q × P == Total: True


## 8. Save the clean data


In [14]:
df.to_csv("../data/clean/clean_cafe_sales.csv", index=False)
print("Saved!")

Saved!


## Summary
| | Before | After |
|---|---|---|
| Rows | 10,000 | 9,514 |
| Missing values | many | 0 |
| Rows removed | – | 486 (4.9%) |
